# CSET485 – AI and Society: Assignment #1
**Roll Number Ends In:** 470
**Dataset:** COMPAS Recidivism Risk Score


## Data Preprocessing
Before any analysis, the dataset was cleaned and preprocessed as follows:
- **Missing Values:** Handled by filtering out invalid scores (`score_text != 'N/A'`) and applying `.dropna()` to the selected subset to ensure a complete case analysis.
- **Categorical Encoding:** One-hot encoding was applied to the categorical features (`sex`, `race`, `c_charge_degree`) using `pd.get_dummies` with `drop_first=True` to avoid the dummy variable trap in our regression models.
- **Feature Scaling:** All numeric features (`age`, `juv_fel_count`, `juv_misd_count`, `juv_other_count`, `priors_count`) were standardized using `StandardScaler` (zero mean, unit variance) fitted only on the training set.
- **Outliers:** Addressed by applying ProPublica's standard validity filters, such as restricting `days_b_screening_arrest` to between -30 and 30 days. This removes erroneous data entry outliers where the arrest date does not align logically with the screening date.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('data/compas-scores-two-years.csv')
df = df.loc[:, ~df.columns.duplicated()]

# ProPublica standard filters to clean data
df = df[(df['days_b_screening_arrest'] <= 30) & 
        (df['days_b_screening_arrest'] >= -30) & 
        (df['is_recid'] != -1) & 
        (df['c_charge_degree'] != 'O') & 
        (df['score_text'] != 'N/A')]

# Select features
features = ['age', 'sex', 'race', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree']

# Target variables (Continuous and Binary)
targets = ['decile_score', 'two_year_recid']
data = df[features + targets].dropna().copy()

# Categorical Encoding (One-Hot Encoding, drop_first to avoid multicollinearity)
data = pd.get_dummies(data, columns=['sex', 'race', 'c_charge_degree'], drop_first=True, dtype=int)

# Train-Test Split (Seed = 470 based on roll number)
X = data.drop(columns=['decile_score', 'two_year_recid'])
y_decile = data['decile_score']
y_recid = data['two_year_recid']

X_train, X_test, y_decile_train, y_decile_test, y_recid_train, y_recid_test = train_test_split(
    X, y_decile, y_recid, test_size=0.30, random_state=470
)

# Feature Scaling (Standardization on numeric columns)
scaler = StandardScaler()
numeric_cols = ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

print(f"Training set size: {X_train.shape[0]}, Test set size: {X_test.shape[0]}")


## Part 1 — Build


In [ ]:
# Model A (Linear Regression for continuous decile_score)
model_a = sm.OLS(y_decile_train, X_train_sm).fit()

# Model B (Logistic Regression for binary two_year_recid)
model_b = sm.Logit(y_recid_train, X_train_sm).fit(disp=0)

# Predictions on test set
pred_a = model_a.predict(X_test_sm)
pred_b = model_b.predict(X_test_sm)

# Thresholds
median_a = pred_a.median()
print(f"Model A median predicted decile score: {median_a:.3f}")

class_a = (pred_a >= median_a).astype(int)
class_b = (pred_b >= 0.5).astype(int)

# Confusion Matrix
cm = confusion_matrix(class_a, class_b)
print("\nConfusion Matrix (Model A vs Model B):")
print("                Model B Low (0)  Model B High (1)")
print(f"Model A Low (0)       {cm[0,0]:<15} {cm[0,1]}")
print(f"Model A High (1)      {cm[1,0]:<15} {cm[1,1]}")

mismatches = cm[0,1] + cm[1,0]
total = len(class_a)
print(f"\nMismatched cases: {mismatches} out of {total} ({(mismatches/total)*100:.2f}%)")


## Part 2 — The Trap


### 2.1: Simpson's Paradox Hunt


In [ ]:
# Splitting by Gender (sex_Male)
sub_a_idx = X_train_sm['sex_Male'] == 1
sub_b_idx = X_train_sm['sex_Male'] == 0

full_model = sm.Logit(y_recid_train, X_train_sm).fit(disp=0)
model_a = sm.Logit(y_recid_train[sub_a_idx], X_train_sm[sub_a_idx].drop(columns=['sex_Male'])).fit(disp=0)
model_b = sm.Logit(y_recid_train[sub_b_idx], X_train_sm[sub_b_idx].drop(columns=['sex_Male'])).fit(disp=0)

print("Coefficient for 'age':")
print(f"Full Model:     {full_model.params['age']:.4f} (p={full_model.pvalues['age']:.4f})")
print(f"Subgroup A (M): {model_a.params['age']:.4f} (p={model_a.pvalues['age']:.4f})")
print(f"Subgroup B (F): {model_b.params['age']:.4f} (p={model_b.pvalues['age']:.4f})")


### 2.2: Omitted-Variable Bias


In [ ]:
# Deliberately dropping 'priors_count'
X_train_reduced = X_train_sm.drop(columns=['priors_count'])
reduced_model = sm.Logit(y_recid_train, X_train_reduced).fit(disp=0)

print("Before (Full) vs After (Reduced) Coefficients:")
predictors_to_check = ['age', 'sex_Male', 'race_Caucasian', 'c_charge_degree_M']
for p in predictors_to_check:
    coef_full = full_model.params[p]
    coef_red = reduced_model.params[p]
    pct_change = ((coef_red - coef_full) / abs(coef_full)) * 100
    print(f"{p:<20} Before: {coef_full:8.4f}  After: {coef_red:8.4f}  Change: {pct_change:6.2f}%")


### 2.3: Adversarial Subset Construction


In [ ]:
pred_probs = full_model.predict(X_test_sm)
test_results = pd.DataFrame({'prob': pred_probs, 'actual': y_recid_test})

# Highly confident but wrong
adv_mask = ((test_results['prob'] >= 0.75) & (test_results['actual'] == 0)) | \
           ((test_results['prob'] <= 0.25) & (test_results['actual'] == 1))

adv_subset = test_results[adv_mask]
print(f"Found {len(adv_subset)} high-confidence wrong predictions ({(len(adv_subset)/len(y_recid_test))*100:.2f}% of test set)")

# Extract Demographics
sample_size = min(50, len(adv_subset))
adv_sample = adv_subset.sample(n=sample_size, random_state=470)
adv_demo = X_test.loc[adv_sample.index].copy()
adv_demo[numeric_cols] = scaler.inverse_transform(adv_demo[numeric_cols])

print("\nAdversarial Subset Mean Demographics (vs Full Test Mean):")
print(f"{'Feature':<20} {'Subset Mean':<12} {'Full Test Mean':<12}")
for col in ['age', 'priors_count', 'sex_Male', 'race_Caucasian']:
    if col in numeric_cols:
        full_mean = scaler.inverse_transform(X_test[numeric_cols])[:, numeric_cols.index(col)].mean()
    else:
        full_mean = X_test[col].mean()
    print(f"{col:<20} {adv_demo[col].mean():<12.2f} {full_mean:<12.2f}")


### 2.4: Fairness Trade-Off Analysis


In [ ]:
group_a_idx = X_test_sm['race_Caucasian'] == 1
group_b_idx = X_test_sm['race_Caucasian'] == 0
fairness_results = []

for t in np.arange(0.05, 1.0, 0.1):
    preds_t = (pred_probs >= t).astype(int)
    
    p_hat_a = preds_t[group_a_idx].mean()
    p_hat_b = preds_t[group_b_idx].mean()
    dp = p_hat_a / p_hat_b if p_hat_b > 0 else np.nan
    
    actual_a = y_recid_test[group_a_idx]
    actual_b = y_recid_test[group_b_idx]
    
    tpr_a = np.sum((preds_t[group_a_idx] == 1) & (actual_a == 1)) / np.sum(actual_a == 1) if np.sum(actual_a == 1) > 0 else 0
    tpr_b = np.sum((preds_t[group_b_idx] == 1) & (actual_b == 1)) / np.sum(actual_b == 1) if np.sum(actual_b == 1) > 0 else 0
    fpr_a = np.sum((preds_t[group_a_idx] == 1) & (actual_a == 0)) / np.sum(actual_a == 0) if np.sum(actual_a == 0) > 0 else 0
    fpr_b = np.sum((preds_t[group_b_idx] == 1) & (actual_b == 0)) / np.sum(actual_b == 0) if np.sum(actual_b == 0) > 0 else 0
    
    eo = min(tpr_a/tpr_b if tpr_b > 0 else 1, fpr_a/fpr_b if fpr_b > 0 else 1)
    
    prec_a = np.sum((actual_a == 1) & (preds_t[group_a_idx] == 1)) / np.sum(preds_t[group_a_idx] == 1) if np.sum(preds_t[group_a_idx] == 1) > 0 else 0
    prec_b = np.sum((actual_b == 1) & (preds_t[group_b_idx] == 1)) / np.sum(preds_t[group_b_idx] == 1) if np.sum(preds_t[group_b_idx] == 1) > 0 else 0
    pp = prec_a / prec_b if prec_b > 0 else 1
    
    fairness_results.append({'Threshold': round(t, 2), 'DP': round(dp, 4), 'EO': round(eo, 4), 'PP': round(pp, 4)})

import matplotlib.pyplot as plt
fairness_df = pd.DataFrame(fairness_results)
print(fairness_df.to_string(index=False))

plt.figure(figsize=(8,5))
plt.plot(fairness_df['DP'], fairness_df['EO'], marker='o')
for i, txt in enumerate(fairness_df['Threshold']):
    plt.annotate(txt, (fairness_df['DP'][i], fairness_df['EO'][i]), xytext=(5,5), textcoords='offset points')
plt.xlabel('Demographic Parity (DP)')
plt.ylabel('Equalized Odds (EO)')
plt.title('Fairness Trade-off: DP vs EO')
plt.grid(True)
plt.show()


## Part 3 — Descriptive Analysis

**1. Comparison of Classifications**
The threshold-based linear regression approach (Model A) and the direct logistic regression approach (Model B) do not identify the exact same individuals as “high risk” within our dataset. According to the test set comparison, there are precisely 286 mismatched cases out of 1852 total test observations, which comprises exactly 15.44% of the data. 
This significant difference arises primarily because the two models rely on fundamentally different mathematical functions to map the independent features to the target variable. Model A utilizes Ordinary Least Squares (OLS) regression, which forecasts a continuous decile score that is completely unbounded. To classify people into the “high risk” category using this method, we are forced to use the median predicted value of 4.525 as our strict decision threshold. On the other hand, Model B utilizes Logistic Regression, which employs a sigmoid function to directly estimate the underlying probability or likelihood of recidivism. This approach inherently constrains all resulting predictions to a strictly bounded range of 0 to 1, and utilizes a standard hardened probability threshold of 0.5 to make classifications. 
Ultimately, the decision boundaries created by the hard thresholds of these two distinct models are not entirely consistent. This is because the linear regression model can generate unbounded predictions that do not represent true probabilities, and it assumes a strictly linear relationship with the decile score rather than fitting a true probability distribution. Consequently, cases lying near the median score of 4.525 in Model A, or near the 0.5 probability boundary in Model B, are the most highly susceptible to being identified differently by the two modeling approaches.

**2. Subgroup Effects (Simpson's Paradox Hunt)**
In the Simpson's Paradox hunt, we isolated a subgroup based on gender (`sex_Male`). While the overall trend did not reverse direction, the magnitude of the effect of `age` on recidivism differed significantly between the subgroups. 
In the full model, the regression coefficient for `age` is **-0.5127** (p-value: 0.0000), indicating a strong negative effect (as age increases, recidivism likelihood decreases). When isolated, the coefficient for Subgroup A (Males) becomes slightly stronger at **-0.5262** (p-value: 0.0000). However, for Subgroup B (Females), the coefficient is noticeably weaker at **-0.4539** (p-value: 0.0000). This indicates that aging serves as a stronger protective factor against recidivism for men than it does for women. 
If this subgroup were ignored, policy recommendations would rely solely on the full model's coefficient of -0.5127. A one-size-fits-all policy might overestimate the protective effect of aging for female offenders and underestimate it for male offenders. Consequently, a criminal justice intervention allocating rehabilitative resources based on age thresholds might prematurely withdraw support for older females, falsely assuming their risk decreases at the exact same rate as males, ultimately leading to inefficient and unfair outcomes.

**3. Omitted-Variable Bias**
For the omitted-variable bias test, the confounding variable `priors_count` was intentionally dropped. The shifts in the coefficients of the remaining predictors are detailed below:
- `age`: Before -0.5127, After -0.3244, Change **36.72%**
- `sex_Male`: Before 0.2877, After 0.4059, Change **41.08%**
- `c_charge_degree_M`: Before -0.1273, After -0.3021, Change **-137.28%**
- `race_Caucasian`: Before -0.0065, After -0.2577, Change **-3893.20%**
The coefficient that shifted the most by far was **`race_Caucasian`**, which experienced a staggering **-3893.20% change**, moving from near zero (-0.0065) to a strong negative effect (-0.2577). 
This drastic shift is highly revealing. It demonstrates that `priors_count` is heavily correlated with race in this dataset. When `priors_count` is omitted, the logistic regression model forces the `race_Caucasian` variable to absorb the omitted variable's predictive power. This reveals that trusting a model without stress-testing for confounds is deeply dangerous; the model falsely attributed a strong behavioral trait (having prior crimes) directly to a demographic trait (race). In high-stakes societal decisions, failing to account for such confounds leads to algorithmic bias and unjust racial profiling.

**4. Detecting Hidden Confounds**
As a practitioner, to detect a hidden confound that is not explicitly present in the data, I would propose conducting a rigorous **Domain Expert Audit coupled with Residual Analysis**. 
In the context of criminal risk scoring, I would extract a stratified sample of the model's most extreme errors (both severe false positives and severe false negatives) based on the residual values. I would then convene a panel of domain experts—such as parole officers, social workers, and public defenders—to qualitatively review these specific case files. These experts have access to unstructured contextual information absent from structured CSVs, such as family support systems, neighborhood policing intensity, or mental health interventions. If the panel consistently identifies a recurring trait among the false positives (e.g., individuals living in highly policed zip codes are systematically over-flagged), it strongly implies a hidden confound (policing bias). This real-world qualitative feedback acts as a critical sanity check, uncovering systemic gaps in the feature space that pure statistical testing cannot organically discover.

**5. Adversarial Subset**
The adversarial subset consists of **87 rows**, which represents **4.70%** of the total test set (1852 cases). These are instances where the model makes highly confident but incorrect predictions (predicting probabilities >= 0.75 for actual 0s, or <= 0.25 for actual 1s). 
A glaring common characteristic shared by this subset is the combination of significantly higher age and a higher number of prior crimes. Specifically, the subset has a **mean age of 43.58** (compared to the full test mean of 34.27) and a **mean `priors_count` of 6.76** (compared to the full test mean of 3.21). 
Statistically, the model fails confidently on these rows because logistic regression assumes a linear relationship between the log-odds of recidivism and the features. The model learned a strong negative coefficient for `age` (-0.5127) and a strong positive coefficient for `priors_count` (inferred from the omitted variable test). When an individual presents extreme values in competing features (e.g., they are much older, which pulls the probability down, but have many priors, which pushes it up), the linear combination creates an extreme logit score. The model confidently misclassifies these individuals because it fails to capture non-linear interactions—such as the reality that an older individual with a long history of past crimes might have "aged out" of crime differently than a younger offender.

**6. Fairness Trade-Offs in Concrete Terms**
The fairness trade-off cost highlights that improving the model's fairness for one group inherently harms another metric, directly impacting real human lives. Looking at the analysis, adjusting the threshold from **t=0.25 to t=0.35** improves Equalized Odds (EO) slightly but causes Demographic Parity (DP) to drop drastically from **0.8910 to 0.7983**. 
In a realistic scenario, consider two individuals up for parole: one African-American and one Caucasian, neither of whom will actually commit another crime. If policymakers decide to maximize Predictive Parity to ensure that a "high-risk" flag means exactly the same thing regardless of race, they might raise the threshold to **t=0.65** (where PP reaches a high of 0.8206). However, doing so plummets Equalized Odds down to **0.5087**. For the real people involved, this means the False Positive Rates become deeply skewed. The African-American individual now faces a disproportionately higher likelihood of being falsely labeled "high risk" compared to the Caucasian individual. Consequently, the African-American individual is unjustly denied parole and remains incarcerated solely due to the mathematical trade-off required to achieve predictive precision.

**7. Defensible Threshold Choice**
I argue that a decision threshold of **t=0.35** is defensible from a fairness standpoint. At this threshold, the model maintains a relatively balanced compromise across competing metrics: **Demographic Parity (DP) sits at 0.7983**, **Equalized Odds (EO) is 0.8190**, and **Predictive Parity (PP) is 0.7534**. This threshold is arguably the most equitable "middle ground" in our sweep, as it ensures that error rates (EO) are somewhat comparable across racial lines while preventing the positive prediction rate (DP) from skewing too heavily toward one demographic. 
The strongest counter-argument a critic could raise is that a Predictive Parity of **0.7534** is too low for high-stakes criminal justice applications. A critic would correctly point out that at t=0.35, the "high-risk" flag is significantly less reliable for one racial group than another. If a judge relies on this score, they are acting on unequal certainty. The critic might argue that a threshold like **t=0.85** is necessary, which pushes PP up to **0.8589**, prioritizing precision. I acknowledge the validity of this argument; if a model's prediction holds different weight depending on a person's race, it fundamentally violates the principle of impartial justice, making low PP a severe liability in the courtroom.

**8. Policy Recommendation**
In the context of the COMPAS recidivism risk dataset, I would strongly recommend prioritizing **Equalized Odds (EO)** over Demographic Parity (DP). Equalized Odds ensures that individuals who do not re-offend have an equal chance of being correctly classified as low-risk, regardless of their race (balancing False Positive Rates). 
Looking at our data, if a policymaker attempts to force near-perfect Demographic Parity (**DP = 0.9804 at t=0.15**), they achieve parity in prediction volumes, but Equalized Odds and Predictive Parity suffer (**PP drops to 0.7484**). 
I am comfortable sacrificing Demographic Parity because DP ignores the underlying base rates of recidivism, which may unfortunately differ between groups due to historical systemic issues (like over-policing). Forcing a model to flag an identical percentage of individuals across groups (DP) when actual recidivism rates differ mathematically requires accepting higher error rates for one demographic. Prioritizing Equalized Odds is a more acceptable sacrifice because it guarantees that the algorithm's *mistakes*—the act of punishing the innocent (false positive) or letting the guilty go (false negative)—are distributed equitably, adhering closer to the legal standard of fairness.


## Part 4 — Reflection

If a policymaker asked me for a one-number answer on whether we should deploy this model, I would say: I recommend conditional deployment under strict human supervision, rather than full autonomous deployment, because the model contains embedded biases that cannot be mathematically resolved by a single threshold.

A single-number answer fails for several critical reasons. First, fairness metrics inherently conflict with one another. As shown in the trade-off chart, it is mathematically impossible to satisfy all fairness criteria simultaneously when base rates differ across groups. Improving Demographic Parity comes at the direct expense of Equalized Odds or Predictive Parity. You cannot tune a single threshold to make the model "fair" for everyone; choosing a threshold is fundamentally a values question, not a data-driven fact. 

Second, the model is highly vulnerable to hidden confounds that remain unknowable without external context. My omitted-variable test demonstrated that dropping a single variable (`priors_count`) caused the coefficient for race to shift by over 3800%. The model simply reallocated the risk of the missing variable onto a demographic proxy. If the model is deployed as a single objective source of truth, it will confidently launder these missing societal variables into racial or gender penalties without anyone realizing it.

Finally, the model confidently fails on predictable subsets of the population. By treating all risk factors as linear and independent, it misses crucial human nuances. For example, my adversarial analysis found 87 cases (nearly 5% of the test set) where the model was highly confident but completely wrong. These were mostly older individuals with many prior offenses. The model simply added the "low risk" of old age to the "high risk" of many priors and produced confident errors, entirely missing the real-world interaction that older habitual offenders often naturally age out of crime. Overall accuracy completely hides these localized performance gaps.

The evidence from the analysis strongly supports these limitations. Sweeping the threshold in Part 2.4 showed that moving from t=0.55 to t=0.35 improves Demographic Parity from 0.51 to 0.79, but it does so by drastically altering the false positive rates for Caucasians. In Part 2.2, dropping the priors variable caused the race coefficient to shift by 3893%. In Part 2.3, the model was confidently wrong on 87 specific defendants.

Ultimately, responsible AI deployment requires transparency about trade-offs, not a single metric or threshold. A predictive model in the justice system should only be a tool to assist domain experts, who can evaluate the hidden confounds and contextual interactions that the algorithm is mathematically blind to.
